<a href="https://colab.research.google.com/github/Changin7/ALURA--Challenge-telecom-X-part-2/blob/main/Telecom_X_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df_clientes = pd.read_csv('datos_filtrados.csv')
df_clientes = df_clientes.drop(columns=['id'])

columnas_internet = [
'seguridad_online', 'backup_online', 'proteccion_dispositivo',
'soporte_tecnico', 'streaming_tv', 'streaming_peliculas'
]

for col in columnas_internet:
df_clientes[col] = df_clientes[col].replace({'Sin servicio de internet': 'No'})

print("Información inicial del dataset:")
df_clientes.info()

In [ ]:
num_vars = df_clientes.select_dtypes(include=["int64", "float64"]).copy()
num_vars["Churn_Num"] = df_clientes["Churn"].map({"Si": 1, "No": 0})

matriz_corr_num = num_vars.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(matriz_corr_num, annot=True, cmap="mako", fmt=".2f")
plt.title("Correlación de Variables Numéricas")
plt.show()

df_clientes = df_clientes.drop(columns=["total_cobrado", "cuentas_diarias"])

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import chi2
from IPython.display import display

df_chi = df_clientes.copy()
df_chi["Churn"] = df_chi["Churn"].map({"Si": 1, "No": 0})

var_categoricas = df_chi.select_dtypes(include=["object"]).columns
var_categoricas = var_categoricas.drop("Churn")

codificador = LabelEncoder()
for col in var_categoricas:
df_chi[col] = codificador.fit_transform(df_chi[col])

X_categoricas = df_chi[var_categoricas]
y_objetivo = df_chi["Churn"]

puntajes_chi, valores_p = chi2(X_categoricas, y_objetivo)

resultados_chi = pd.DataFrame({
"Característica": var_categoricas,
"Puntaje_Chi2": puntajes_chi,
"P_Valor": valores_p
}).sort_values(by="Puntaje_Chi2", ascending=False)

print("Resultados del Test Chi-Cuadrado:")
display(resultados_chi)

df_clientes = df_clientes.drop(columns=["genero", "servicio_telefono", "multiples_lineas"])

In [ ]:

df_clientes["Churn"] = df_clientes["Churn"].map({"Si": 1, "No": 0})


X_datos = df_clientes.drop('Churn', axis=1)
y_etiquetas = df_clientes['Churn']


X_codificado = pd.get_dummies(X_datos, drop_first=True)

from sklearn.model_selection import train_test_split

X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(
X_codificado,
y_etiquetas,
test_size=0.25,
stratify=y_etiquetas,
random_state=42
)

print(f"Datos para entrenar: {X_entrenamiento.shape[0]} registros")
print(f"Datos para probar: {X_prueba.shape[0]} registros")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

#Regresión Logística
print("="*40)
print("  MODELO 1: REGRESIÓN LOGÍSTICA")
print("="*40)
modelo_logistico = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
modelo_logistico.fit(X_entrenamiento, y_entrenamiento)
pred_log = modelo_logistico.predict(X_prueba)

print(classification_report(y_prueba, pred_log))
ConfusionMatrixDisplay.from_predictions(y_prueba, pred_log, cmap="Blues", display_labels=['No Abandona', 'Sí Abandona'])
plt.title("Matriz de Confusión - Regresión Logística")
plt.show()

#Bosque Aleatorio
print("\n" + "="*40)
print("  MODELO 2: RANDOM FOREST")
print("="*40)
modelo_bosque = RandomForestClassifier(random_state=42, class_weight="balanced")
modelo_bosque.fit(X_entrenamiento, y_entrenamiento)
pred_bosque = modelo_bosque.predict(X_prueba)

print(classification_report(y_prueba, pred_bosque))
ConfusionMatrixDisplay.from_predictions(y_prueba, pred_bosque, cmap="Greens", display_labels=['No Abandona', 'Sí Abandona'])
plt.title("Matriz de Confusión - Random Forest")
plt.show()

#Árbol de Decisión
print("\n" + "="*40)
print("  MODELO 3: ÁRBOL DE DECISIÓN")
print("="*40)
modelo_arbol = DecisionTreeClassifier(max_depth=6, random_state=42, class_weight="balanced")
modelo_arbol.fit(X_entrenamiento, y_entrenamiento)
pred_arbol = modelo_arbol.predict(X_prueba)

print(classification_report(y_prueba, pred_arbol))
ConfusionMatrixDisplay.from_predictions(y_prueba, pred_arbol, cmap="Oranges", display_labels=['No Abandona', 'Sí Abandona'])
plt.title("Matriz de Confusión - Árbol de Decisión")
plt.show()